# NEET Mock Intelligence — cloud analysis (no PC / no API key)

**Use this notebook from an iPad, phone or any browser.** It runs the open-weight models in Google Colab, not on your device. You only provide the 30 mock PDFs and the actual NEET 2026 PDF; the notebook builds the question bank, analyses every extracted question, compares every NEET question to the mock bank and produces the detailed PDF report plus an audit workbook.

### Before running

1. Choose **File → Save a copy in Drive**. The notebook includes a self-contained copy of the source code, so the run does not need to read your GitHub repository later. You can make that repository private after saving your copy.
2. Choose **Runtime → Change runtime type → T4 GPU** if it is offered. Free Colab GPU access is not guaranteed. If no GPU is available, the notebook offers a smaller CPU fallback which is much slower.
3. Run the cells in order. The first cell asks permission to connect to your Google Drive. Upload PDFs in the next two cells.

### What the report contains

For each NEET question: original question text, every accepted mock match with **Test number + mock question number**, one of seven relationship categories, shared concept, exact differences, solution-method comparison, evidence, confidence, and verification status. It also lists questions with no meaningful match. A second PDF catalogues all extracted mock questions and their concepts/solutions. An Excel workbook preserves the full audit trail.

### Important free-runtime limits

Google Colab free compute is temporary: GPU availability and usage limits vary, and a free runtime can run for up to 12 hours depending on availability and usage. The notebook checkpoints its database to Drive after every paper, analysis batch and NEET match. If Colab disconnects, reconnect, rerun setup/model cells, then rerun the analysis cell; completed work is skipped. Model weights may need to download again after a new runtime.

This is a free **notebook workflow**, not a permanently-running website. A permanently available web app with guaranteed GPU capacity and unlimited processing cannot realistically be provided at $0. Your PDFs and analysis outputs are stored in your Google Drive; inference runs inside Google's Colab runtime, without sending questions to a paid third-party model API.

In [ ]:
# 1) Connect Drive, unpack the embedded app, install libraries and restore saved progress.
from google.colab import drive, files
from pathlib import Path
import os, io, base64, zipfile, shutil, sys, subprocess, textwrap

drive.mount('/content/drive')

WORKSPACE = Path('/content/drive/MyDrive/NEET-Mock-Intelligence')
for folder in ["input/mock_papers", "input/mock_solutions", "input/neet_2026", "data/page_images", "data/cache", "reports"]:
    (WORKSPACE / folder).mkdir(parents=True, exist_ok=True)

PROJECT_DIR = Path('/content/neet_mock_intelligence')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_BUNDLE_B64 = """..."""
project_zip = base64.b64decode(PROJECT_BUNDLE_B64)
with zipfile.ZipFile(io.BytesIO(project_zip)) as zf:
    zf.extractall(PROJECT_DIR)

# Persist inputs, page previews, embeddings and final files on Drive. Keep the active
# SQLite file on Colab's fast local disk and mirror it to Drive with SQLite's backup API.
RUNTIME_DIR = Path('/content/neet_mock_runtime')
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
DB_LOCAL = RUNTIME_DIR / 'neet_matcher.sqlite3'
DB_DRIVE = WORKSPACE / 'data/neet_matcher.sqlite3'
if not DB_LOCAL.exists() and DB_DRIVE.exists():
    shutil.copy2(DB_DRIVE, DB_LOCAL)

os.environ['NEET_WORKSPACE_DIR'] = str(WORKSPACE)
os.environ['NEET_DATA_DIR'] = str(WORKSPACE / 'data')
os.environ['NEET_PAGE_IMAGE_DIR'] = str(WORKSPACE / 'data/page_images')
os.environ['NEET_CACHE_DIR'] = str(WORKSPACE / 'data/cache')
os.environ['NEET_DB_PATH'] = str(DB_LOCAL)
os.environ['NEET_DB_MIRROR_PATH'] = str(DB_DRIVE)
os.environ['OCR_ENABLED'] = 'true'
os.environ['RENDER_PAGE_IMAGES'] = 'true'
os.environ['EXPECTED_MOCK_QUESTIONS'] = '180'
os.environ['EXPECTED_NEET_QUESTIONS'] = '180'

# PDF text extraction and OCR support.
subprocess.run(['bash', '-lc', 'apt-get update -qq && apt-get install -y -qq tesseract-ocr'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_DIR/'requirements-colab.txt')], check=True)

sys.path.insert(0, str(PROJECT_DIR))
from core import config
from core.database import init_db, checkpoint_database
init_db()
checkpoint_database()

MOCK_DIR = config.MOCK_DIR
SOLUTION_DIR = config.SOLUTION_DIR
NEET_DIR = config.NEET_DIR
NEET_PDF = NEET_DIR / 'NEET_2026.pdf'
print('Workspace:', WORKSPACE)
print('Existing mock PDFs:', len(list(MOCK_DIR.glob('*.pdf'))))
print('NEET paper present:', NEET_PDF.exists())
print('Saved database restored:', DB_LOCAL.exists())


In [ ]:
# 2) Upload your 30 mock-test PDFs together. Optional solution PDFs may be included;
# name them with the word "solution", "answer" or "key" so they are sorted correctly.
from google.colab import files
import shutil

existing_mocks = sorted(MOCK_DIR.glob('*.pdf'))
if len(existing_mocks) >= 30:
    print(f'{len(existing_mocks)} mock PDFs already exist in your Drive workspace. Reusing them; no re-upload needed.')
else:
    print('Select all 30 mock question-paper PDFs in the file picker. Optional separate solution PDFs can be selected too.')
    uploaded = files.upload()
    mock_saved, solution_saved = [], []
    for filename, file_bytes in uploaded.items():
        if not filename.lower().endswith('.pdf'):
            print('Skipped non-PDF:', filename)
            continue
        lowered = filename.lower()
        target_dir = SOLUTION_DIR if any(term in lowered for term in ('solution', 'answer', 'answerkey', 'answer_key', 'key')) else MOCK_DIR
        target = target_dir / Path(filename).name
        target.write_bytes(file_bytes)
        (solution_saved if target_dir == SOLUTION_DIR else mock_saved).append(target.name)
    del uploaded
    print(f'Saved {len(mock_saved)} mock PDFs and {len(solution_saved)} solution PDFs to Drive.')

mock_files = sorted(MOCK_DIR.glob('*.pdf'))
solution_files = sorted(SOLUTION_DIR.glob('*.pdf'))
print(f'Mock PDFs ready: {len(mock_files)} / 30')
print('Solution PDFs ready:', len(solution_files))
for p in mock_files:
    print(' •', p.name)
if len(mock_files) != 30:
    raise RuntimeError('The folder must contain exactly 30 mock question PDFs before analysis. Upload the missing papers or remove accidental PDFs from MyDrive/NEET-Mock-Intelligence/input/mock_papers/.')


In [ ]:
# 3) Upload the real NEET 2026 question paper. It is saved as NEET_2026.pdf in Drive.
from google.colab import files

if NEET_PDF.exists():
    print('NEET_2026.pdf already exists in your Drive workspace. Reusing it.')
else:
    print('Choose the actual NEET 2026 question-paper PDF (not a mock).')
    neet_upload = files.upload()
    pdf_items = [(name, data) for name, data in neet_upload.items() if name.lower().endswith('.pdf')]
    if len(pdf_items) != 1:
        raise RuntimeError(f'Please upload exactly one NEET 2026 PDF. Received {len(pdf_items)} PDFs.')
    NEET_PDF.write_bytes(pdf_items[0][1])
    del neet_upload
    print('Saved:', NEET_PDF)

if not NEET_PDF.exists() or NEET_PDF.stat().st_size == 0:
    raise RuntimeError('NEET 2026 PDF is missing or empty.')
print('NEET PDF ready:', NEET_PDF.name, f'({NEET_PDF.stat().st_size/1024/1024:.1f} MB)')


In [ ]:
# 4) Extract every PDF and verify the full paper counts before downloading large AI models.
# This catches missing questions and poor OCR early, without spending GPU time.
from core.workflow import ingest_folder, ingest_one_file
from core.database import get_papers, get_questions, checkpoint_database

print('Reading mock PDFs and retaining original page references…')
mock_ingest = ingest_folder(config.MOCK_DIR, 'mock')
if list(config.SOLUTION_DIR.glob('*.pdf')):
    solution_ingest = ingest_folder(config.SOLUTION_DIR, 'solution')
else:
    solution_ingest = {'imported': [], 'skipped': [], 'warnings': []}
neet_ingest = ingest_one_file(NEET_PDF, 'neet', 1)
checkpoint_database()

all_papers = get_papers()
mock_papers = [p for p in all_papers if p['kind'] == 'mock']
mock_questions = get_questions('mock')
neet_questions = get_questions('neet')
print(f'Mock PDFs extracted: {len(mock_papers)} / 30')
print(f'Mock questions extracted: {len(mock_questions)} / 5,400')
print(f'NEET questions extracted: {len(neet_questions)} / 180')
for paper in mock_papers:
    if paper['question_count'] != 180:
        print(f"QUESTION COUNT WARNING — {paper['filename']}: {paper['question_count']} questions")
odd_papers = [p for p in mock_papers if p['question_count'] != 180]
if len(mock_papers) != 30 or odd_papers or len(mock_questions) != 5400 or len(neet_questions) != 180:
    raise RuntimeError('Extraction preflight failed. Do not continue to model setup. Check the PDF files, scans, OCR and page layout. The extracted data is checkpointed in Drive; replace a bad PDF in the Drive input folder and rerun this cell.')
print('Extraction preflight passed. All 5,400 mock questions and 180 NEET questions are indexed.')

In [ ]:
# 5) Start Ollama inside the Colab cloud runtime and download free open-weight models.
# No Ollama/GitHub/Hugging Face API key is required. Model files are downloaded to this
# temporary Colab runtime and do not need to be uploaded to GitHub or Drive.
import os, subprocess, time, requests, torch

has_gpu = torch.cuda.is_available()
if has_gpu:
    print('GPU detected:')
    subprocess.run(['nvidia-smi'], check=False)
    PRIMARY_MODEL = 'qwen3.5:9b'      # multimodal primary analyst
    VERIFIER_MODEL = 'deepseek-r1:8b' # independent reasoning verifier
    MODEL_TIER = 'GPU tier: Qwen3.5 9B + DeepSeek-R1 8B'
else:
    print('WARNING: No GPU is attached. Free GPU availability varies. You can select Runtime → Change runtime type → T4 GPU and rerun this cell.')
    PRIMARY_MODEL = 'qwen3.5:4b'
    VERIFIER_MODEL = 'deepseek-r1:1.5b'
    MODEL_TIER = 'CPU fallback: Qwen3.5 4B + DeepSeek-R1 1.5B (much slower and less capable)'

os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
os.environ['OLLAMA_CONTEXT_LENGTH'] = '16384'
os.environ['OLLAMA_NUM_PARALLEL'] = '1'
os.environ['OLLAMA_MAX_LOADED_MODELS'] = '1'
os.environ['OLLAMA_KEEP_ALIVE'] = '0'
os.environ['OLLAMA_FLASH_ATTENTION'] = '1'

if not shutil.which('ollama'):
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)

ollama_log = open('/content/ollama.log', 'w')
ollama_process = subprocess.Popen(['ollama', 'serve'], stdout=ollama_log, stderr=subprocess.STDOUT, start_new_session=True)
for _ in range(60):
    try:
        if requests.get('http://127.0.0.1:11434/api/tags', timeout=2).ok:
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError('Ollama did not start. See /content/ollama.log in the Colab Files panel.')

print('Selected model tier:', MODEL_TIER)
for model in [PRIMARY_MODEL, VERIFIER_MODEL]:
    print('Downloading model:', model)
    subprocess.run(['ollama', 'pull', model], check=True)

# Set the app's model endpoints BEFORE importing the workflow modules.
os.environ['LLM_BASE_URL'] = 'http://127.0.0.1:11434/v1'
os.environ['LLM_API_KEY'] = 'ollama'
os.environ['LLM_MODEL'] = PRIMARY_MODEL
os.environ['VERIFY_BASE_URL'] = 'http://127.0.0.1:11434/v1'
os.environ['VERIFY_API_KEY'] = 'ollama'
os.environ['VERIFY_MODEL'] = VERIFIER_MODEL
os.environ['VISION_BASE_URL'] = 'http://127.0.0.1:11434/v1'
os.environ['VISION_API_KEY'] = 'ollama'
os.environ['VISION_MODEL'] = PRIMARY_MODEL
os.environ['LLM_TEMPERATURE'] = '0.1'
os.environ['LLM_TIMEOUT'] = '600'
os.environ['LLM_MAX_RETRIES'] = '4'
os.environ['LLM_BATCH_SIZE'] = '3'
os.environ['TOP_CANDIDATES'] = '30'
os.environ['MAX_MATCHES_PER_NEET_QUESTION'] = '10'
os.environ['EMBEDDING_MODEL'] = 'BAAI/bge-m3'
os.environ['ENABLE_EMBEDDINGS'] = 'true'
os.environ['ENABLE_RERANKER'] = 'false'
os.environ['VISION_MAX_IMAGES'] = '6'

# The configuration module may have been imported during the setup cell; reload it so
# the selected Colab models/paths are active for this run.
import importlib
import core.config as config
importlib.reload(config)
from core import llm, workflow, database
importlib.reload(llm)
importlib.reload(database)
importlib.reload(workflow)
from core.database import init_db, checkpoint_database
init_db()
checkpoint_database()
print('AI team is ready. Primary:', PRIMARY_MODEL, '| Verifier:', VERIFIER_MODEL, '| Vision:', PRIMARY_MODEL)
print('Embedding model: BAAI/bge-m3 (local). Candidate reranker: disabled to conserve free GPU memory.')


In [ ]:
# 6) Run the complete company-style workflow. This cell is resumable.
# If Colab disconnects or a run stops, reconnect, run setup/model cells again, then rerun this cell.
from tqdm.auto import tqdm
from IPython.display import display, Markdown
from core import config
from core.workflow import run_full_analysis
from core.database import checkpoint_database

mock_files = sorted(config.MOCK_DIR.glob('*.pdf'))
if len(mock_files) != 30:
    raise RuntimeError(f'Expected 30 mock PDFs; found {len(mock_files)}.')
if not (config.NEET_DIR / 'NEET_2026.pdf').exists():
    raise RuntimeError('Upload the NEET_2026.pdf in Step 3 first.')

bar = tqdm(total=100, desc='AI Operations Manager')
def show_progress(current, total, message):
    total = max(int(total), 1)
    value = max(0, min(100, int(current * 100 / total)))
    bar.n = value
    bar.set_description(str(message)[:90])
    bar.refresh()

try:
    RESULT = run_full_analysis(config.MOCK_DIR, config.SOLUTION_DIR, config.NEET_DIR / 'NEET_2026.pdf', show_progress)
    checkpoint_database()
finally:
    bar.close()

print('Analysis finished.')
print('Detailed NEET comparison PDF:', RESULT['pdf_path'])
print('Full mock-question catalogue PDF:', RESULT['catalogue_path'])
print('Audit workbook:', RESULT['workbook_path'])
print('Warnings:', len(RESULT.get('warnings', [])))
if RESULT.get('warnings'):
    display(Markdown('### Review warnings\n' + '\n'.join('- ' + str(x) for x in RESULT['warnings'][:60])))


In [ ]:
# 7) Package the final deliverables and download them to your iPad/phone.
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
from IPython.display import display, Markdown

report_paths = [Path(RESULT['pdf_path']), Path(RESULT['catalogue_path']), Path(RESULT['workbook_path'])]
missing = [str(p) for p in report_paths if not p.exists()]
if missing:
    raise RuntimeError('Some report files were not created: ' + ', '.join(missing))

bundle_path = config.DATA_DIR / 'NEET_Mock_Intelligence_Final_Deliverables.zip'
with ZipFile(bundle_path, 'w', ZIP_DEFLATED) as zf:
    for p in report_paths:
        zf.write(p, arcname=p.name)

print('Saved all three deliverables in:', config.DATA_DIR)
for p in report_paths:
    print(' •', p.name, f'({p.stat().st_size/1024:.0f} KB)')
print('Download bundle:', bundle_path)
files.download(str(bundle_path))


## If the runtime disconnects

Your PDFs, question analyses, matches, page images and report files stay in your Drive workspace. Reopen your saved notebook copy and run the cells again. The upload cells detect existing PDFs and reuse them. The model download cell may need to run again because model weights are stored in the temporary Colab runtime. The analysis cell resumes using the Drive database checkpoint.

**Do not delete** `MyDrive/NEET-Mock-Intelligence/data/neet_matcher.sqlite3` while a run is incomplete; that file contains the progress checkpoint.